In [1]:
import pandas as pd
import numpy as np

from linearmodels import PanelOLS

In [2]:
data = pd.read_csv('cleaned_school_data.csv')  

In [3]:
print(data.groupby("Year")["mean_scale_score"].agg(["mean", "std", "min", "max"]).round(3))

         mean     std    min    max
Year                               
2018  599.896  11.460  555.0  647.0
2019  599.042  11.667  553.0  646.0
2022  599.976  11.671  554.0  650.0


In [4]:
# adjust % poverty to a percentage scale
data['poverty_percentage'] = (data['% Poverty'] * 100).round(3)

In [5]:
# # preliminary DiD variables
# data['post'] = (data['Year'] == 2022).astype(int)
# data['treated_cont'] = data['post'] * data['poverty_percentage']

### Individual Grade Datasets

In [ ]:
# grade splits

data_3 = data[
    (data['Grade'] == '3') & 
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]


data_4 = data[
    (data['Grade'] == '4') &  
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_5 = data[
    (data['Grade'] == '5') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_6 = data[
    (data['Grade'] == '6') &   
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_7 = data[
    (data['Grade'] == '7') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_8 = data[
    (data['Grade'] == '8') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]



### Verifying Parallel Trends Assumption in the Pre Period

#### All Grades

In [8]:
# Time-placebo test: use 2018 and 2019 as "pre" period
# this is a method to test the parallel trends assumption by checking for any pre-existing trends in the outcome variable before the treatment period
pre_data = data[data['Year'].isin([2018,2019])].copy() # new pre data
pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage'] # new placebo interaction term for treated

In [9]:
pre_data = pre_data.set_index(['DBN', 'Year']) # set index for PanelOLS, it requires a multi-index with entity and time dimensions

In [10]:
# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model = PanelOLS(
    dependent=pre_data['mean_scale_score'],
    exog=pre_data[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [11]:
print(f"Placebo test coefficient: {parallel_trends_model.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: -0.0016, p-value: 0.5982


```python
Placebo test coefficient: -0.0009, p-value: 0.8371
``` 

This is great!!!! Near zero trend in pre-period, parallel trends assumption met

#### Same thing for individual grades

In [16]:
pre_data3 = data_3[data_3['Year'].isin([2018,2019])].copy() # new pre data
pre_data3['fake_post'] = (pre_data3['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data3['fake_treated_cont'] = pre_data3['fake_post'] * pre_data3['poverty_percentage'] # new placebo interaction term for treated

pre_data3 = pre_data3.set_index(['DBN', 'Year'])


# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model3 = PanelOLS(
    dependent=pre_data3['mean_scale_score'],
    exog=pre_data3[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data3['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

print(f"Placebo test coefficient: {parallel_trends_model3.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model3.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: 0.0230, p-value: 0.0182


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [17]:
results = {}

for i in range(3, 9):
    data = globals()[f"data_{i}"]
    
    pre_data = data[data['Year'].isin([2018, 2019])].copy()
    pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int)
    pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage']
    
    pre_data = pre_data.set_index(['DBN', 'Year'])
    
    model = PanelOLS(
        dependent=pre_data['mean_scale_score'],
        exog=pre_data[['fake_treated_cont']],
        entity_effects=True,
        time_effects=True,
        weights=pre_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    results[i] = model
    
    print(f"Dataset {i} → Coef: {model.params['fake_treated_cont']:.4f}, "
          f"p-value: {model.pvalues['fake_treated_cont']:.4f}")

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 3 → Coef: 0.0230, p-value: 0.0182


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 4 → Coef: 0.0289, p-value: 0.0036


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 5 → Coef: -0.0058, p-value: 0.5272


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 6 → Coef: -0.0395, p-value: 0.0011
Dataset 7 → Coef: -0.0396, p-value: 0.0040


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 8 → Coef: -0.0007, p-value: 0.9570


A very scary output to see when assessing parallel trends. Luckily, I am not the first person to pursue a project using DID and have assumptions fail. The difficulty with this model is the alleged binary of "yes" or "no" when talking about parallel trends violation. It is unrealistic to expect consistency when dealing woth real-world data - the world is imperfect and things tend to fluctuate. <br> 

The question is, how do you move forward from this? Rambachan & Roth (2023) have a different methodological approach involving a sensitivity analysis. Testing for parallel trends is already problematic to begin with, but still may have some insight into pre-trends. This paper suggests that instead of passing or failing the assumption to assess how large of a violation does it need to be to affect results. The researchers developed an HonestDID package, which allows for sensitivity analysis towards the magnitude of the assumption violation.

### Model 1: Pooled Ordinary Least Squares Model

Pooled OLS treats panel data as if it is one cross-sectional piece of data. Essentially, this is a baseline model where time and group fixed effects are ignored. <br>
https://www.geeksforgeeks.org/artificial-intelligence/pooled-ols-regression/